# Limpieza y exploración del dataset de empleados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

## 1. Carga de datos
Se carga el archivo original manteniendo los nombres de columnas en inglés.

In [ ]:
df = pd.read_csv("hr.csv")

## 2. Exploración inicial
Se visualizan algunas filas para entender la estructura del dataset.

In [ ]:
display(df.head(2))
display(df.tail(2))
display(df.sample(2, random_state=42))

## 3. Dimensiones del dataset
Número de filas y columnas.

In [ ]:
print(df.shape)

## 4. Información general y valores nulos
Se revisan tipos de datos y nulos.

In [ ]:
df.info()
df.isna().sum()

## 5. Limpieza de nombres de columnas
Se pasa todo a snake_case sin cambiar el idioma.

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(r'([a-z])([A-Z])', r'\1_\2', regex=True)
    .str.replace(" ", "_")
    .str.lower()
)

## 6. Conversión de variables numéricas
Se asegura que algunas columnas sean numéricas.

In [ ]:
df["age"] = pd.to_numeric(df["age"], errors="coerce").astype("Int64")

## 7. Limpieza de variables categóricas
Se homogenizan los textos.

In [ ]:
for col in df.select_dtypes(include=["object","string"]).columns:
    df[col] = df[col].astype(str).str.lower().str.strip().str.replace(" ", "_")

## 8. Conversión a booleanos
Se convierten columnas tipo sí/no a booleanos.

In [ ]:
bool_map = {"yes":True,"no":False,"y":True,"n":False}

for col in ["attrition","overtime"]:
    if col in df.columns:
        df[col] = df[col].map(bool_map)

## 9. Eliminación de columnas irrelevantes
Columnas constantes o sin valor analítico.

In [ ]:
df = df.drop(columns=[
    col for col in ["over18","standardhours","employeecount"]
    if col in df.columns
])

## 10. Estadística descriptiva
Resumen de variables numéricas.

In [ ]:
df.describe(include="number").T

## 11. Correlación de variables salariales
Relación entre ingresos.

In [ ]:
cols = [c for c in ["dailyrate","monthlyincome","monthlyrate"] if c in df.columns]
df[cols].corr(numeric_only=True)

## 12. Análisis de attrition
Se estudia la deserción según el manager actual.

In [ ]:
if {"yearswithcurrmanager","attrition"}.issubset(df.columns):
    df.groupby("yearswithcurrmanager")["attrition"].mean()

## 13. Revisión final
Se comprueba el resultado final del dataset.

In [ ]:
display(df.head())
df.dtypes